In [1]:
import pandas as pd
from datetime import datetime
import os
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
# Load the CSV file into a pandas DataFrame
cPath = os.path.dirname(os.path.realpath('./New/'))
file_path = os.path.realpath('./Combined_Terminal_Label.csv')
print('File: ', file_path)
myData = pd.read_csv(file_path, parse_dates=['DateTime'], date_format='%Y-%m-%d %H:%M:%S')
myData.set_index('DateTime', inplace=True)
print(myData.head(3))
print(myData.tail(3))

File:  /home/homayoun/Dropbox/Projects/VisualCode/Python/CAE/New/KWH_based/Combined_Terminal_Label.csv
                     KWH11    KW  Year     Delta     label
DateTime                                                  
2014-07-01 00:14:00  357.5  1430  2014  0.005000  sOffPeak
2014-07-01 00:29:00  362.0  1448  2014 -0.012778  sOffPeak
2014-07-01 00:44:00  350.5  1402  2014 -0.006667  sOffPeak
                     KWH11   KW  Year     Delta     label
DateTime                                                 
2024-12-31 23:29:00  211.0  844  2024  0.008889  wOffPeak
2024-12-31 23:44:00  219.0  876  2024 -0.027778  wOffPeak
2024-12-31 23:59:00  194.0  776  2024  0.000000  wOffPeak


In [3]:
# Convert to daily usage by summing KWH11 for each day and keep only KWH11 column
myData_daily = myData.resample('D').agg({'KWH11': 'sum'})
myData_yearly_avg = myData_daily.resample('YE').agg({'KWH11': 'mean'})

print(myData_daily.head(2))
print(myData_daily.tail(2))
print(myData_yearly_avg.head(2))
print(myData_yearly_avg.tail(2))
print(f"\nOriginal data points: {len(myData)}")
print(f"Daily data points: {len(myData_daily)}")

              KWH11
DateTime           
2014-07-01  28243.5
2014-07-02  29073.0
              KWH11
DateTime           
2024-12-30  15499.5
2024-12-31  17205.5
                   KWH11
DateTime                
2014-12-31  25319.385870
2015-12-31  23377.141096
                   KWH11
DateTime                
2023-12-31  19681.190411
2024-12-31  21083.689891

Original data points: 368264
Daily data points: 3837


In [4]:
# Create subplot with daily usage and yearly average on the same plot
fig_combined = go.Figure()

# Add daily usage trace
fig_combined.add_trace(go.Scatter(
    x=myData_daily.index,
    y=myData_daily['KWH11'],
    mode='lines',
    name='Daily Usage',
    line=dict(color='blue', width=1),
    hovertemplate='Date: %{x}<br>Daily Usage: %{y:.2f} KWh<extra></extra>'
))

# Add yearly average trace
fig_combined.add_trace(go.Scatter(
    x=myData_yearly_avg.index,
    y=myData_yearly_avg['KWH11'],
    mode='lines+markers',
    name='Yearly Average',
    line=dict(color='red', width=3),
    marker=dict(size=8),
    hovertemplate='Year: %{x|%Y}<br>Yearly Avg: %{y:.2f} KWh<extra></extra>'
))

fig_combined.update_layout(
    title='Daily Electricity Usage with Yearly Averages',
    xaxis_title='Date',
    yaxis_title='Usage (KWh)',
    hovermode='x unified',
    legend=dict(x=0.02, y=0.08)
)

fig_combined.show()

In [5]:
# Plot with area fill and color by label
iYear = 2024
data = myData[myData['Year'] == iYear].reset_index()
fig = px.area(
    data,
    # myData[(myData['Year'] >= 2015) & (myData['Year'] <= 2020)], 
    # myData, 
    x='DateTime', 
    y='KWH11', 
    color='label', 
    hover_data={'label': False, 'DateTime': False, 'KWH11': ':.2f'},
    title=f'Categorized Electricity Usage in {iYear} (KWh)'
    )
fig.update_layout(hovermode='x unified')
fig.show()

In [6]:
# Fit a quadratic trend to annual average usage: y = a*year^2 + b*year + c
years = myData_yearly_avg.index.year.astype(float).to_numpy()
annual_usage = myData_yearly_avg["KWH11"].to_numpy()

coeffs = np.polyfit(years, annual_usage, 2)
myData_yearly_avg["KWH11_QuadraticTrend"] = np.polyval(coeffs, years)

print(f"Quadratic fit: y = {coeffs[0]:.6f}*year^2 + {coeffs[1]:.2f}*year + {coeffs[2]:.2f}")

# Optional: quick visualization
fig_yearly_trend = go.Figure()
fig_yearly_trend.add_trace(go.Scatter(
    x=myData_yearly_avg.index, y=myData_yearly_avg["KWH11"],
    mode="lines+markers", name="Actual"
))
fig_yearly_trend.add_trace(go.Scatter(
    x=myData_yearly_avg.index, y=myData_yearly_avg["KWH11_QuadraticTrend"],
    mode="lines", name="Quadratic Trend", line=dict(dash="dash")
))
fig_yearly_trend.update_layout(
    title="Annual Usage with Quadratic Trend",
    xaxis_title="Year",
    yaxis_title="Average Usage (KWh)",
    hovermode="x unified"
)
fig_yearly_trend.show()

Quadratic fit: y = 208.659342*year^2 + -843039.65*year + 851544456.28


In [7]:
# Project average annual usage for 2025 and 2026 using the existing quadratic fit
projection_years = np.array([2025.0, 2026.0])
projected_usage = np.polyval(coeffs, projection_years)

projection_df = pd.DataFrame({
    "Year": projection_years.astype(int),
    "Projected_Avg_KWH11": projected_usage
})

print(projection_df.to_string(index=False))

 Year  Projected_Avg_KWH11
 2025         22884.982696
 2026         25124.330487


In [12]:
# Repeat yearly trend figure and include projected usages for 2025 and 2026
fig_yearly_trend_proj = go.Figure()

# Use integer years on the x-axis to avoid visual shifts from year-end timestamps
year_x = myData_yearly_avg.index.year.astype(int)
projection_years = np.array([2025, 2026], dtype=int)
projection_labels = [f"{value:.2f} KWh" for value in projected_usage]

fig_yearly_trend_proj.add_trace(go.Scatter(
    x=year_x,
    y=myData_yearly_avg["KWH11"],
    mode="lines+markers",
    name="Actual Yearly Average",
    line=dict(color="royalblue", width=2),
    hovertemplate="Year: %{x}<br>Actual Avg: %{y:.2f} KWh<extra></extra>"
))

fig_yearly_trend_proj.add_trace(go.Scatter(
    x=year_x,
    y=myData_yearly_avg["KWH11_QuadraticTrend"],
    mode="lines",
    name="Quadratic Trend",
    line=dict(color="firebrick", width=3, dash="dash"),
    hovertemplate="Year: %{x}<br>Trend Avg: %{y:.2f} KWh<extra></extra>"
))

fig_yearly_trend_proj.add_trace(go.Scatter(
    x=projection_years,
    y=projected_usage,
    mode="markers+text",
    name="Projected (2025-2026)",
    marker=dict(color="seagreen", size=10, symbol="diamond"),
    text=projection_labels,
    textposition="top center",
    hovertemplate="Year: %{x}<br>Projected Avg: %{y:.2f} KWh<extra></extra>"
))

fig_yearly_trend_proj.update_layout(
    title="Yearly Average Usage with Quadratic Trend and Projections",
    xaxis_title="Year",
    yaxis_title="Usage (KWh)",
    xaxis=dict(
        type="linear",
        dtick=1,
        range=[int(year_x.min()) - 0.5, int(projection_years.max()) + 0.5]
    ),
    hovermode="x unified"
 )

fig_yearly_trend_proj.show()